In [2]:
import pandas as pd

sentiment = pd.read_csv("fear_greed_index.csv")
trades = pd.read_csv("historical_data (1).csv")

In [3]:
# doing basic data inspection
sentiment.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2644 entries, 0 to 2643
Data columns (total 4 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   timestamp       2644 non-null   int64 
 1   value           2644 non-null   int64 
 2   classification  2644 non-null   object
 3   date            2644 non-null   object
dtypes: int64(2), object(2)
memory usage: 82.8+ KB


In [4]:
sentiment.describe()

,timestamp,value
count,2.644000e+03,2644.000000
mean,1.631899e+09,46.981089
std,6.597967e+07,21.827680
min,1.517463e+09,5.000000
25%,1.574811e+09,28.000000
50%,1.631900e+09,46.000000
75%,1.688989e+09,66.000000
max,1.746164e+09,95.000000


In [5]:
trades.info()
trades.describe()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 211224 entries, 0 to 211223
Data columns (total 16 columns):
 #   Column            Non-Null Count   Dtype  
---  ------            --------------   -----  
 0   Account           211224 non-null  object 
 1   Coin              211224 non-null  object 
 2   Execution Price   211224 non-null  float64
 3   Size Tokens       211224 non-null  float64
 4   Size USD          211224 non-null  float64
 5   Side              211224 non-null  object 
 6   Timestamp IST     211224 non-null  object 
 7   Start Position    211224 non-null  float64
 8   Direction         211224 non-null  object 
 9   Closed PnL        211224 non-null  float64
 10  Transaction Hash  211224 non-null  object 
 11  Order ID          211224 non-null  int64  
 12  Crossed           211224 non-null  bool   
 13  Fee               211224 non-null  float64
 14  Trade ID          211224 non-null  float64
 15  Timestamp         211224 non-null  float64
dtypes: bool(1), float64(

,Execution Price,Size Tokens,Size USD,Start Position,Closed PnL,Order ID,Fee,Trade ID,Timestamp
count,211224.000000,2.112240e+05,2.112240e+05,2.112240e+05,211224.000000,2.112240e+05,211224.000000,2.112240e+05,2.112240e+05
mean,11414.723350,4.623365e+03,5.639451e+03,-2.994625e+04,48.749001,6.965388e+10,1.163967,5.628549e+14,1.737744e+12
std,29447.654868,1.042729e+05,3.657514e+04,6.738074e+05,919.164828,1.835753e+10,6.758854,3.257565e+14,8.689920e+09
min,0.000005,8.740000e-07,0.000000e+00,-1.433463e+07,-117990.104100,1.732711e+08,-1.175712,0.000000e+00,1.680000e+12
25%,4.854700,2.940000e+00,1.937900e+02,-3.762311e+02,0.000000,5.983853e+10,0.016121,2.810000e+14,1.740000e+12
50%,18.280000,3.200000e+01,5.970450e+02,8.472793e+01,0.000000,7.442939e+10,0.089578,5.620000e+14,1.740000e+12
75%,101.580000,1.879025e+02,2.058960e+03,9.337278e+03,5.792797,8.335543e+10,0.393811,8.460000e+14,1.740000e+12
max,109004.000000,1.582244e+07,3.921431e+06,3.050948e+07,135329.090100,9.014923e+10,837.471593,1.130000e+15,1.750000e+12


In [6]:
# inconsistencies are there in col namestrades.columns = trades.columns.str.lower().str.replace(" ", "_")
sentiment.columns = sentiment.columns.str.lower()
trades.columns = trades.columns.str.lower().str.replace(" ", "_")


In [8]:
# fix the timestamps.  the time in trades is in ms.
trades['time'] = pd.to_datetime(trades['timestamp'], unit='ms')
sentiment['date'] = pd.to_datetime(sentiment['date'])


In [9]:
print(trades['time'].min(), trades['time'].max()) #check 
print(sentiment['date'].min(), sentiment['date'].max())

2023-03-28 10:40:00 2025-06-15 15:06:40
2018-02-01 00:00:00 2025-05-02 00:00:00


In [11]:
print("Sentiment shape:", sentiment.shape)
print("Trades shape:", trades.shape)

print(sentiment.columns)
print(trades.columns)


Sentiment shape: (2644, 4)
Trades shape: (211224, 17)
Index(['timestamp', 'value', 'classification', 'date'], dtype='object')
Index(['account', 'coin', 'execution_price', 'size_tokens', 'size_usd', 'side',
       'timestamp_ist', 'start_position', 'direction', 'closed_pnl',
       'transaction_hash', 'order_id', 'crossed', 'fee', 'trade_id',
       'timestamp', 'time'],
      dtype='object')


In [12]:
# check missing val
print(sentiment.isnull().sum())
print(trades.isnull().sum())

timestamp         0
value             0
classification    0
date              0
dtype: int64
account             0
coin                0
execution_price     0
size_tokens         0
size_usd            0
side                0
timestamp_ist       0
start_position      0
direction           0
closed_pnl          0
transaction_hash    0
order_id            0
crossed             0
fee                 0
trade_id            0
timestamp           0
time                0
dtype: int64


In [14]:
sentiment = sentiment.drop_duplicates()
trades = trades.drop_duplicates()
print("Duplicates removed:", ...)

Duplicates removed: Ellipsis


In [16]:
#daily level
trades['date'] = trades['time'].dt.date
sentiment['date'] = pd.to_datetime(sentiment['date']).dt.date

In [17]:
#merging
df = trades.merge(sentiment[['date','classification']], on='date', how='left')
print(df['classification'].isnull().sum())

26961


In [18]:
# now there null. performing null analysis by dropping rows
# find which dates r missing
print("Trades:", trades['date'].min(), trades['date'].max())
print("Sentiment:", sentiment['date'].min(), sentiment['date'].max())

Trades: 2023-03-28 2025-06-15
Sentiment: 2018-02-01 2025-05-02


In [19]:
missing_dates = df[df['classification'].isnull()]['date'].unique()
print(len(missing_dates))
print(missing_dates[:10])

1
[datetime.date(2025, 6, 15)]


In [20]:
# came to know That single missing day (2025-06-15) has many trades.. thats why it showed nulls as 26961
# only 1 day is there. so we are droppig that day
df = df.dropna(subset=['classification'])

In [21]:
# save the cleaned dataset as new dataset for analysis
df.to_csv("cleaned_trades.csv", index=False)

In [28]:
import os
import shutil

# move cleaned file → processed folder
shutil.move("cleaned_trades.csv", "processed/cleaned_trades.csv")

# move raw files → raw folder
shutil.move("fear_greed_index.csv", "raw/fear_greed_index.csv")
shutil.move("historical_data (1).csv", "raw/historical_data.csv")

'raw/historical_data.csv'